**“This notebook explores different modeling approaches (RandomForest, XGBoost) to predict real estate prices. It includes preprocessing, cross-validation, and submission file generation.”****

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Import the libraries

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Extract and load the data + preview

In [ ]:
# load files  train.csv and test.csv
train_csv = "/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
test_csv  = "/kaggle/input/house-prices-advanced-regression-techniques/test.csv"
df_train = pd.read_csv(train_csv)
df_test = pd.read_csv(test_csv)
df_test['SalePrice']=0
df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
print("Train shape:", df.shape)


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

# Quick EDA: statistics, missing values, correlation

In [ ]:
df['SalePrice'].describe().round(2)

In [ ]:
# Visualize the target column 'SalePrice'
df['SalePrice'].plot(kind='hist', bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of SalePrice")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

In [ ]:
#% missing valuer per columns
missing = df.isna().mean().sort_values(ascending=False)
missing.head(20)

In [ ]:
# Copier le DataFrame
data = df.copy()
# Numeric columns and correlation with SalePrice
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = data[num_cols].corr()['SalePrice'].sort_values(ascending=False)
print("\nCorrelation of numerical features with SalePrice:")
print(corr_with_target.head(20))

In [ ]:
# Correlation Matrix of Numerical Variables
plt.figure(figsize=(20, 15))

sns.heatmap(
    df[num_cols].corr(),
    annot=True,  # Display correlation values
    fmt=".2f",   # Format the annotations to 2 decimal places
    cmap='vlag', # Diverging color map (good for correlation)
    center=0,    # Ensure 0 is the center point for the color scale
    linewidths=1.5, # Add lines between cells for better separation
    linecolor='black', # Color of the lines
    cbar_kws={'shrink': .75, 'orientation': 'vertical', 'label': 'Coefficient de Corrélation'} # Customize the color bar
)

# Improve title visibility and position
plt.title(
    " Correlation Matrix of Numerical Variables",
    fontsize=20,
    fontweight='bold',
    pad=20 # Add padding between the title and the plot
)

# Rotate axis labels for better readability if variable names are long
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# Adjust layout to prevent labels from being cut off
plt.tight_layout()

plt.show()

In [ ]:
# missing values
missing = data.isna().sum().sort_values(ascending=False)
missing.head(20)

In [ ]:
# Separate numerical and categorical features
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('SalePrice')  # remove the target column

cat_cols = data.select_dtypes(include=['object']).columns.tolist()

print(f"Number of numerical columns: {len(num_cols)}")
print(f"Number of categorical columns: {len(cat_cols)}")

In [ ]:
# Identify NAN columns
# Percentage of missing values per column
missing_percent = data.isna().mean() * 100
print(missing_percent.sort_values(ascending=False).head(20))

In [ ]:
cols_to_drop = missing_percent[missing_percent > 50].index
data.drop(cols_to_drop.tolist(), axis=1, inplace=True, errors='ignore')
data.reset_index(drop=True, inplace=True)
data.columns


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select numerical columns
num_data = data[num_cols]

# Figure size
plt.figure(figsize=(15, len(num_cols) * 3))

# Create a boxplot for each numerical column
for i, col in enumerate(num_data.columns, 1):
    plt.subplot(len(num_data.columns), 1, i)
    sns.boxplot(x=num_data[col], color="skyblue")
    plt.title(f"Distribution and outliers: {col}", fontsize=11)
    plt.xlabel("")
    plt.tight_layout()

plt.show()



In [ ]:
# Create a complete pipeline with model

# 1 Outlier detection using IQR
# ===============================
num_cols = data.select_dtypes(include=[np.number]).columns.tolist()
outliers_dict = {}

for col in num_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5*IQR
    upper_bound = Q3 + 1.5*IQR
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    if len(outliers) > 0:
        outliers_dict[col] = (lower_bound, upper_bound)
        print(f"{col} : {len(outliers)} outliers detected")

# 2 Outlier correction using Winsorization
for col, (lower, upper) in outliers_dict.items():
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

print("\nAll extreme values have been corrected (Winsorization).")

#
# 3 Post-correction verification
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    outlier_count = len(df[(df[col] < lower) | (df[col] > upper)])
    if outlier_count > 0:
        print(f"{col} : {outlier_count} remaining outliers")
    else:
        print(f"{col} : no remaining outliers")

# 4 Visualization after correction (example: SalePrice)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(df['SalePrice'], bins=50, kde=True, color='skyblue')
plt.title("Distribution of SalePrice after correction")
plt.xlabel("Sale Price ($)")

plt.subplot(1,2,2)
sns.boxplot(x=df['BsmtFinSF1'], color='lightgreen')
plt.title("Boxplot of SalePrice after correction")
plt.xlabel("Sale Price ($)")

plt.tight_layout()
plt.show()


In [ ]:
#Verifier
print("missing values after traitment:")
print(data.isna().sum())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# --- Séparation features / target ---
X = data.drop(columns=['SalePrice'])
y = data['SalePrice']

# --- Identifier les colonnes numériques et catégorielles ---
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# --- Pipelines numériques et catégorielles ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# --- Combinaison via ColumnTransformer ---
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# --- Transformation des données ---
X_processed = preprocessor.fit_transform(X)

# --- Récupération des noms de colonnes ---
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
ohe_cols = ohe.get_feature_names_out(cat_cols) if cat_cols else []
all_cols = num_cols + list(ohe_cols)

# --- Création du DataFrame final ---
X_final = pd.DataFrame(X_processed, columns=all_cols)

# --- y reste inchangé ---
y_final = y


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score
import xgboost as xgb

# --- Séparation propre des features et de la cible ---
X = data.drop(columns=['SalePrice'])
y = data['SalePrice']

# --- Identifier les types de colonnes ---
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# --- Pipelines individuels ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# --- Préprocesseur combiné ---
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# --- Pipeline complet avec XGBoost ---
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        objective='reg:squarederror'  # ← indispensable pour les nouvelles versions de XGBoost
    ))
])

# --- Validation croisée ---
scores_xgb = cross_val_score(
    xgb_pipeline,
    X, y,
    cv=3,
    scoring='neg_root_mean_squared_error'
)

print("RMSE CV (XGBoost) :", -scores_xgb.round(2))
print("RMSE moyen :", -scores_xgb.mean().round(2))


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# --- Grille d’hyperparamètres ---
param_grid = {
    'model__n_estimators': [100, 300, 500],
    'model__max_depth': [3, 5, 7, 10],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.7, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.9, 1.0],
    'model__gamma': [0, 0.1, 0.3],        # régularisation
    'model__reg_lambda': [1, 5, 10]       # L2 regularization
}

# --- Recherche aléatoire ---
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_grid,
    n_iter=20,                              # 20 combinaisons aléatoires
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# --- Entraînement / optimisation ---
xgb_search.fit(X, y)

# --- Résultats ---
print("\n✅ Meilleurs hyperparamètres :", xgb_search.best_params_)
print("📉 Meilleur RMSE CV :", round(-xgb_search.best_score_, 2))


In [ ]:
# --- Installations (si nécessaire) ---
!pip install catboost lightgbm xgboost -q

# --- Imports ---
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# --- Définir les sous-modèles ---
estimators = [
    ('xgb', xgb.XGBRegressor(
        n_estimators=500, max_depth=5, learning_rate=0.1,
        subsample=0.7, colsample_bytree=0.9, random_state=42,
        objective='reg:squarederror'
    )),
    ('lgb', lgb.LGBMRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.9, random_state=42
    )),
    ('cat', CatBoostRegressor(
        iterations=500, depth=6, learning_rate=0.05,
        verbose=0, random_state=42
    ))
]

# --- Préprocessing : numérique + catégoriel ---
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()



# --- Pipelines ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))  # ✅ dense output
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# --- Stacking ---
stack_model = StackingRegressor(
    estimators=[
        ('xgb', xgb.XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.1, subsample=0.7, colsample_bytree=0.9, random_state=42, objective='reg:squarederror')),
        ('lgb', lgb.LGBMRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.9, random_state=42)),
        ('cat', CatBoostRegressor(iterations=500, depth=6, learning_rate=0.05, verbose=0, random_state=42))
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)

stack_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', stack_model)
])

stack_pipeline.fit(X, y)

# --- Validation croisée ---
scores = cross_val_score(stack_pipeline, X, y, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)

print("RMSE CV (Stacking) :", -scores.round(2))
print("RMSE moyen :", round(-scores.mean(), 2))


In [ ]:
import numpy as np

# --- Préparer le test set ---
X_test = df_test.copy()

# Ajouter les colonnes manquantes (présentes dans X mais pas dans X_test)
for col in X.columns:
    if col not in X_test.columns:
        X_test[col] = np.nan

# Retirer les colonnes supplémentaires (sécurité) et réordonner
X_test = X_test[X.columns]

# --- Prédictions sur le test set ---

y_pred = stack_pipeline.predict(X_test)

# --- Vérifier quelques prédictions ---
print("Aperçu des prédictions :", y_pred[:10])


In [ ]:
# --- 5. Créer le fichier de soumission ---
submission = pd.DataFrame({
    "Id": df_test["Id"],
    "SalePrice": y_pred  # ✅ utiliser la bonne variable
})

submission.to_csv("submission.csv", index=False)
print("✅ Fichier 'submission.csv' prêt pour Kaggle.")
